<a href="https://colab.research.google.com/github/cojocarucosmin/AICourseDev/blob/main/Metadata_Document_Extractor_using_OCR_an_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Parsing and extracting inforrmation from documents using OCR an ChatGPT**

### Install required libraries

In [1]:
!pip install -q pymupdf pytesseract pillow openai gradio
!apt-get install -y -qq tesseract-ocr
!apt-get install -y -qq libtesseract-dev

### Gradio app to upload on demand documents and parse them with ChatGPT

In [3]:
# @title
import gradio as gr
import os
import json
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import pandas as pd
import openai
import tempfile
import shutil

# --- TEXT + OCR EXTRACTION ---
def extract_text_and_images_from_pdf(pdf_path):
    results = []
    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()
        images = page.get_images(full=True)
        image_texts = []

        for img_index, img in enumerate(images, start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))
            ocr_text = pytesseract.image_to_string(image)
            image_texts.append({
                "image_index": img_index,
                "ocr_text": ocr_text.strip()
            })

        results.append({
            "page_num": page_num,
            "text": text.strip(),
            "image_texts": image_texts
        })

    doc.close()
    return results

# --- OPENAI METADATA EXTRACTION ---
def invoke_openai_model(system_prompt, user_content, model, temperature, log):
    try:
        response = openai.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
            temperature=temperature
        )
        metadata_text = response.choices[0].message.content

        json_start = metadata_text.find('{')
        json_end = metadata_text.rfind('}') + 1
        if json_start == -1 or json_end == -1:
            raise ValueError("❌ No JSON object found in GPT response.")

        metadata_json = json.loads(metadata_text[json_start:json_end])
        return metadata_json

    except Exception as e:
        log.append(f"❌ OpenAI or JSON parsing error: {e}")
        return None

# --- MAIN PIPELINE ---
def run_pipeline(api_key, model, temp, system_prompt, uploaded_files, progress=gr.Progress(track_tqdm=True)):
    openai.api_key = api_key
    metadata_list = []
    logs = []

    temp_dir = tempfile.mkdtemp()
    excel_path = os.path.join(temp_dir, "metadata_output.xlsx")

    try:
        for file in progress.tqdm(uploaded_files, desc="Processing PDFs"):
            try:
                file_path = file.name
                logs.append(f"📥 Processing file: {file.name}")

                parsed_data = extract_text_and_images_from_pdf(file_path)
                user_input = json.dumps(parsed_data, ensure_ascii=False, indent=2)
                metadata = invoke_openai_model(system_prompt, user_input, model, temp, logs)

                if metadata:
                    metadata["source_file"] = os.path.basename(file.name)
                    metadata_list.append(metadata)
                    logs.append(f"✅ Done: {file.name}")
                else:
                    logs.append(f"⚠️ Skipped: {file.name}")

            except Exception as e:
                logs.append(f"❌ Error: {file.name} — {e}")
                continue

        df = pd.DataFrame(metadata_list)
        df.to_excel("metadata_output.xlsx", index=False)

        return "metadata_output.xlsx", "\n".join(logs)

    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

# --- GRADIO UI ---

with gr.Blocks(title="📚 AI PDF Metadata Extractor") as demo:
    gr.Markdown("## 📚 AI PDF Metadata Extractor by AI Academy")
    gr.Markdown("Upload your documents, set your prompt, and extract structured metadata with ChatGPT.")

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            system_prompt = gr.Textbox(label="🧠 System Prompt", lines=23, value="""
You are an expert in document metadata extraction for bibliometric and literature review purposes.
Extract the following metadata fields from these documents in a consolidated JSON format:
1. Title
2. Journal
3. DOI
4. Authors
5. Publication Year
6. Keywords
7. Abstract
8. Summary
9. Conclusions
10. Advantages/Disadvantages
11. Explainable AI
12. Next Steps
13. Region
14. Dataset Size
15. Types of Data Used
16. Types of Companies
17. Period Analyzed
18. Models/Approach Used
19. Innovative Algorithms
20. General Applicability
21. Performance for Each Model
""")
            api_key = gr.Textbox(label="🔐 OpenAI API Key", type="password")
            model = gr.Dropdown(["gpt-4o", "gpt-4", "gpt-3.5-turbo"], value="gpt-4o", label="🧠 OpenAI Model")
            temp = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label="🎛️ Temperature")

        with gr.Column(scale=1):
            uploaded = gr.File(label="📄 Upload PDF(s)", file_types=[".pdf"], file_count="multiple", height=150)
            submit_btn = gr.Button("🚀 Extract Metadata", variant="primary")
            excel_output = gr.File(label="📥 Download Excel", interactive=False, file_types=[".xlsx"])
            log_output = gr.Textbox(label="🪵 Logs", lines=8, interactive=False)

    submit_btn.click(
        fn=run_pipeline,
        inputs=[api_key, model, temp, system_prompt, uploaded],
        outputs=[excel_output, log_output]
    )

demo.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://75038dee33c87c5138.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Batch Processing of documents from Drive

### Configure batch processing parameters

In [13]:
from google.colab import userdata

# 🚀 USER CONFIGURATION (EDIT THIS CELL ONLY)

# 📂 Root folder in Google Drive (must contain Input Docs / Output Docs / Results)
BASE_PATH = '/content/drive/My Drive/_Profi/_AI L&D/Docs'

# 🔁 Set to True to force reprocess all files, even if metadata already exists
RUN_ALL = False

# 🔐 OpenAI Settings
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
OPENAI_MODEL = 'gpt-4o'
OPENAI_TEMPERATURE = 0.0
OPENAI_MAX_TOKENS = 4096  # Optional: for controlling response size

### Configure the system prompt - what do you need to extract

In [ ]:
# 🧠 METADATA SYSTEM PROMPT (editable if needed)
SYSTEM_PROMPT = """
You are an expert in document metadata extraction for bibliometric and literature review purposes.
Extract the following metadata fields from these documents in a consolidated JSON format:
1. Title (string)
2. Journal (string)
3. DOI (string)
4. Authors (list of strings)
5. Publication Year (integer)
6. Keywords (list of strings)
7. Abstract (string)
8. Summary (string)
9. Conclusions (string)
10. Advantages/Disadvantages (dictionary)
11. Explainable AI (Yes/No)
12. Next Steps (string)
13. Region (list of strings)
14. Dataset Size (string)
15. Types of Data Used (list: e.g., financial, legal, news, etc.)
16. Types of Companies (list: e.g., SMEs, large enterprises, listed companies etc.)
17. Period Analyzed (range)
18. Models/Approach Used (list of strings)
19. Innovative Algorithms (Yes/No)
20. General Applicability (string)
21. Performance for Each Model (list of dictionaries, e.g., [{"model": "XGBoost", "recall": 0.85}, ...])
"""

### Pre-process documents to extract relevant content

In [18]:
# @title
# 🚫 DO NOT EDIT – PDF to JSON parsing (text + image OCR)

import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import os, json
import openai
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Derived paths
INPUT_DOCS = os.path.join(BASE_PATH, 'Input Docs')
OUTPUT_DOCS = os.path.join(BASE_PATH, 'Output Docs')
RESULTS_FOLDER = os.path.join(BASE_PATH, 'Results')
os.makedirs(OUTPUT_DOCS, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

METADATA_JSON = os.path.join(RESULTS_FOLDER, 'Consolidated_Metadata.json')
METADATA_XLSX = os.path.join(RESULTS_FOLDER, 'Consolidated_Metadata.xlsx')

# Configure OpenAI
openai.api_key = OPENAI_API_KEY

def extract_text_and_images_from_pdf(pdf_path):
    """Extract native text and OCR text from images inside a PDF."""
    results = []
    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()

        images = page.get_images(full=True)
        image_texts = []

        for img_index, img in enumerate(images, start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))

            # OCR with pytesseract
            ocr_text = pytesseract.image_to_string(image)
            image_texts.append({
                "image_index": img_index,
                "ocr_text": ocr_text.strip()
            })

        results.append({
            "page_num": page_num,
            "text": text.strip(),
            "image_texts": image_texts
        })

    doc.close()
    return results

def save_results_to_json(results, filename, destination_folder):
    """Save extracted results to a JSON file in the Output Docs folder."""
    os.makedirs(destination_folder, exist_ok=True)
    json_filename = os.path.splitext(os.path.basename(filename))[0] + ".json"
    json_path = os.path.join(destination_folder, json_filename)

    with open(json_path, "w", encoding="utf-8") as json_file:
        json.dump(results, json_file, ensure_ascii=False, indent=4)

    print(f"✅ Saved parsed JSON: {json_path}")
    return json_filename

def process_pdfs_from_directory(source_folder, output_folder, results_folder):
    """Process PDFs from Input Docs and save JSONs + summary Excel."""
    summary = []
    os.makedirs(results_folder, exist_ok=True)

    for filename in os.listdir(source_folder):
        if not filename.lower().endswith(".pdf"):
            continue

        json_filename = os.path.splitext(filename)[0] + ".json"
        json_path = os.path.join(output_folder, json_filename)

        if os.path.exists(json_path):
            print(f"⏭️ Skipping {filename}: already parsed.")
            continue

        pdf_path = os.path.join(source_folder, filename)
        results = extract_text_and_images_from_pdf(pdf_path)
        save_results_to_json(results, filename, output_folder)

        summary.append({
            "pdf_file": filename,
            "json_file": json_filename,
            "pages": len(results),
        })

    # Save summary to Excel
    if summary:
        df_summary = pd.DataFrame(summary)
        excel_path = os.path.join(results_folder, "processing_summary.xlsx")
        df_summary.to_excel(excel_path, index=False)
        print(f"\n📊 Summary saved to Excel: {excel_path}")
    else:
        print("\n📂 No new PDFs processed.")

# ✅ RUN THIS to parse any new PDFs in Input Docs
process_pdfs_from_directory(
    source_folder=INPUT_DOCS,
    output_folder=OUTPUT_DOCS,
    results_folder=RESULTS_FOLDER
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏭️ Skipping The significance of financial and non-financial information in  insolvency risk detection.pdf: already parsed.
⏭️ Skipping PREDICTION OF CORPORATE BANKRUPTCY IN ROMANIA THROUGH THE USE OF LOGISTIC REGRESSION.pdf: already parsed.
⏭️ Skipping BALANCED BAGGING WITH EXPECTATION MAXIMIZATION.pdf: already parsed.
⏭️ Skipping Diagnostic model of the risk of bankruptcy.pdf: already parsed.

📂 No new PDFs processed.


### Extract metadata with ChatGPT based on prompt instructions

In [19]:
# @title
# 🚫 DO NOT EDIT – SETUP: Mount Drive, prepare folders and OpenAI call + metadata logic

def invoke_openai_model(system_prompt, user_content):
    """Call OpenAI GPT model with system prompt and content."""
    try:
        response = openai.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
            temperature=OPENAI_TEMPERATURE
        )
        metadata_text = response.choices[0].message.content
        metadata_json = json.loads(metadata_text[metadata_text.find('{'):metadata_text.rfind('}') + 1])
        return metadata_json
    except Exception as e:
        print(f"❌ OpenAI Error: {e}")
        return None

def load_existing_metadata(path):
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as file:
                return json.load(file)
        except:
            print("⚠️ Metadata file exists but could not be read. Starting fresh.")
    return []

# 🚫 DO NOT EDIT – Metadata extraction pipeline

def process_json_files(source_folder, metadata_path, run_all=False):
    """Process OCR+text JSONs and extract metadata with OpenAI."""
    if run_all and os.path.exists(metadata_path):
        os.remove(metadata_path)
        print("🗑️ Deleted existing metadata — reprocessing all files.")

    consolidated = load_existing_metadata(metadata_path)
    already_processed = {entry.get("source_file") for entry in consolidated if "source_file" in entry}

    processed_count = 0
    skipped_count = 0
    failed_files = []

    for filename in os.listdir(source_folder):
        filepath = os.path.join(source_folder, filename)

        if not filename.endswith(".json") or not os.path.isfile(filepath):
            continue
        if not run_all and filename in already_processed:
            print(f"⏭️ Already processed: {filename}")
            skipped_count += 1
            continue

        try:
            with open(filepath, "r", encoding="utf-8") as file:
                doc_content = json.load(file)
            user_input = json.dumps(doc_content, ensure_ascii=False, indent=2)

            print(f"\n📄 Extracting metadata for: {filename}")
            metadata = invoke_openai_model(SYSTEM_PROMPT, user_input)

            if metadata:
                metadata["source_file"] = filename
                consolidated.append(metadata)

                with open(metadata_path, "w", encoding="utf-8") as file:
                    json.dump(consolidated, file, ensure_ascii=False, indent=4)

                print(f"✅ Saved: {filename}")
                processed_count += 1
            else:
                failed_files.append(filename)

        except Exception as e:
            print(f"⚠️ Failed to process {filename}: {e}")
            failed_files.append(filename)

    print(f"\n✅ Done: {processed_count} processed | ⏭️ {skipped_count} skipped | ❌ {len(failed_files)} failed")
    return pd.DataFrame(consolidated)

# ✅ RUN THIS to execute the metadata extraction
df_metadata = process_json_files(
    source_folder=OUTPUT_DOCS,
    metadata_path=METADATA_JSON,
    run_all=RUN_ALL
)

if not df_metadata.empty:
    df_metadata.to_excel(METADATA_XLSX, index=False)
    print(f"\n📊 Metadata saved to Excel: {METADATA_XLSX}")
    display(df_metadata.head())
else:
    print("⚠️ No metadata extracted.")


⏭️ Already processed: The significance of financial and non-financial information in  insolvency risk detection.json
⏭️ Already processed: PREDICTION OF CORPORATE BANKRUPTCY IN ROMANIA THROUGH THE USE OF LOGISTIC REGRESSION.json
⏭️ Already processed: BALANCED BAGGING WITH EXPECTATION MAXIMIZATION.json
⏭️ Already processed: Diagnostic model of the risk of bankruptcy.json

✅ Done: 0 processed | ⏭️ 4 skipped | ❌ 0 failed

📊 Metadata saved to Excel: /content/drive/My Drive/_Profi/_AI L&D/Docs/Results/Consolidated_Metadata.xlsx


,Title,Journal,DOI,Authors,Publication Year,Keywords,Abstract,Summary,Conclusions,Advantages/Disadvantages,...,Region,Dataset Size,Types of Data Used,Types of Companies,Period Analyzed,Models/Approach Used,Innovative Algorithms,General Applicability,Performance for Each Model,source_file
0,The significance of financial and non-financia...,Procedia Economics and Finance,10.1016/S2212-5671(15)00834-5,"[Mironiuc Marilena, Taran Alina]",2015,"[insolvency, financial information, multiple d...",Insolvency places under uncertainty the premis...,The study investigates the role of financial a...,The study shows the prevalence of financial in...,{'Advantages': 'The study provides a comprehen...,...,[Romania],20 companies,"[financial, non-financial]",[listed companies],2009-2013,"[multiple discriminant analysis, logistic regr...",No,The study's findings are applicable to underst...,"[{'model': 'Multiple Discriminant Analysis', '...",The significance of financial and non-financia...
1,Prediction of Corporate Bankruptcy in Romania ...,Not specified,Not specified,"[Brîndescu Daniel, Goleț Ionuț]",2023,"[bankruptcy, financial statement analysis, eco...",The purpose of this paper is to test whether d...,The study evaluates the use of logistic regres...,"The fixed assets ratio, fixed assets turnover ...",{'Advantages': 'The model provides a practical...,...,[Romania],"4,327 companies",[financial],"[SMEs, large enterprises]",2008-2012,[Logistic Regression],No,The model is expected to maintain its accuracy...,"[{'model': 'Logistic Regression', 'in-sample a...",PREDICTION OF CORPORATE BANKRUPTCY IN ROMANIA ...
2,Balanced Bagging with Expectation Maximization...,Revista Economica,,[Claudiu Clement],2022,"[bankruptcy, machine learning, classification]",Bankruptcy prediction models are widely used b...,This paper investigates the effect of two impu...,The experimental results show that the Expecta...,{'Advantages': 'Balanced Bagging with Expectat...,...,[Romania],"More than 20,000 companies",[financial],[Romanian companies],2016-2019,"[Balanced Bagging, Logistic Regression, Decisi...",Yes,The study shows that financial statements data...,[{'model': 'Balanced Bagging with Expectation ...,BALANCED BAGGING WITH EXPECTATION MAXIMIZATION...
3,Diagnostic model of the risk of bankruptcy,Procedia Economics and Finance,10.1016/S2212-5671(14)00632-7,[Bircea Ioana],2014,"[Diagnosis, Insolvency, Risks]","In Romania, on the background of the economic ...",The study develops a financial diagnosis model...,Financial diagnosis submitted is checked only ...,{'Advantages': 'The model provides a quick and...,...,[Romania],34 companies,[financial],[small companies],Not specified,"[Financial diagnosis model, Score-based analysis]",No,The model is applicable to small companies in ...,[],Diagnostic model of the risk of bankruptcy.json
